In [ ]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Literal
from langchain_openai import ChatOpenAI
from pydantic import BaseModel,Field
from dotenv import load_dotenv


In [ ]:
load_dotenv()

In [ ]:
model=ChatOpenAI(model='gpt-4o-mini')

In [ ]:
class Schema(BaseModel):
    sentiment:Literal['positive','negative']=Field(description='Sentiment of the review')

In [ ]:
structured_model=model.with_structured_output(Schema)


In [ ]:
prompt=f'What is the sentiment of the following review-The software is too good'
structured_model.invoke(prompt).sentiment

In [ ]:
class ReviewState(TypedDict):
    review:str
    sentiment:Literal['positive','negative']
    diagnosis:dict
    response:str
    

In [ ]:
def find_sentiment(state:ReviewState)->ReviewState:
    prompt=f'For the following review find out the sentiment \n {state['review']}'
    sentiment=structured_model.invoke(prompt).sentiment
    return {'sentiment':sentiment}
def check_sentiment(state:ReviewState)->Literal['positive_response','run_diagnosis']:
    if state['sentiment']=='positive':
        return 'positive_response'
    else:
        return 'run_diagnosis'
def positive_response(state:ReviewState)->ReviewState:
    prompt=f'Write a warm thank you message in response to the review:\n\n{state['review']}'
    response=model.invoke(prompt).content
    return {'response':response}
def run_diagnosis(state:ReviewState)->ReviewState:
    prompt=f'diagnose this negative review:\n \n {state['review']}\n Return issue_type,tone,and urgency'
    response=structured_model2.invoke(prompt)
    return {'diagnosis':response.model_dump()}
def negative_response(state:ReviewState)->ReviewState:
    diagnosis=state['diagnosis']
    prompt=f'the user had a {diagnosis['issue_type']} issue,sounded {diagnosis['tone']},and marked the urgency as {diagnosis['urgency']}'
    response=model.invoke(prompt).content
    return {'response':response}

In [ ]:

class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review')
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description='The emotional tone expressed by the user')
    urgency: Literal["low", "medium", "high"] = Field(description='How urgent or critical the issue appears to be')

structured_model2=model.with_structured_output(DiagnosisSchema)


In [ ]:
graph=StateGraph(ReviewState)
graph.add_node('find_sentiment',find_sentiment)
graph.add_node('positive_response',positive_response)
graph.add_node('run_diagnosis',run_diagnosis)
graph.add_node('negative_response',negative_response)
graph.add_edge(START,'find_sentiment')
graph.add_conditional_edges('find_sentiment',check_sentiment)
graph.add_edge('positive_response',END)
graph.add_edge('run_diagnosis','negative_response')
graph.add_edge('negative_response',END)
workflow=graph.compile()
graph.compile()

In [ ]:
initial_state={'review':'the product was really good'}
final_state=workflow.invoke(initial_state)